# Exercise 6: Imitation Learning

**Topics:** Behavior Cloning | DAgger 

**Duration:** ~1 hour | **GPU:** Not strictly necessary

## Environment & Expert Policy


 ###   Environment
 We use a custom 2-D continuous-control task (`SCurve2DEnv`). An agent must navigate from a start position to a goal along an S-shaped reference curve. The state is $[x, y, v_x, v_y]$ and the action is a 2-D acceleration $[a_x, a_y]$ clipped to $\pm a_\text{max}$. The observation also includes the goal position and several lookahead reference points so the policy can anticipate the curve ahead.

### How the Expert Policy Works

The expert is a **PD (Proportional–Derivative) tracking controller** with lookahead. It does not learn anything — it is a hand-designed controller that has access to the reference path.

At every time step it:

1. **Picks a target point** — the *furthest* lookahead reference point included in the observation (this gives it a preview of where the curve is heading).
2. **Computes a desired velocity** $v_\text{des}$ — a fixed-speed vector pointing along the path tangent at the target, so the agent is encouraged to move forward along the curve.
3. **Applies PD control** to produce an acceleration:

$$a = k_p \, (\text{target} - \text{pos}) \;+\; k_d \, (v_\text{des} - v)$$

   - The **proportional term** ($k_p$) steers the agent toward the target point.
   - The **derivative term** ($k_d$) dampens oscillations by penalising the difference between the current velocity and the desired velocity.

4. **Clips** the resulting acceleration to the action limits $[-a_\text{max},\; a_\text{max}]$.

This controller is far from perfect — it can overshoot on tight curves and does not optimise cumulative reward — but it reliably follows the S-curve and serves as a reasonable "expert" for imitation learning experiments.

**Source of randomness.** The expert policy itself is completely **deterministic**: the same observation always produces the same action. All variation across episodes comes from the environment's `reset`, which adds small Gaussian noise to the initial position (`pos_noise = 0.05`) and initial velocity (`vel_noise = 0.05`). These perturbed starting conditions cause the expert to trace slightly different trajectories each episode, providing diversity in the demonstration dataset.


The cell below defines:

- **`SCurveConfig`** — all tuneable parameters (dynamics limits, path geometry, reward weights, etc.).
- **`SCurve2DEnv`** — a Gymnasium-compatible environment that simulates 2-D double-integrator dynamics along an S-shaped reference curve.
- **`expert_policy`** — a hand-crafted PD tracking controller (see explanation below the code).

> No need to implement anything here, but we suggest to take a look at how the environment and expert policy are defined.


In [ ]:
import numpy as np
from dataclasses import dataclass

try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    # fallback to gym if gymnasium isn't available
    import gym
    from gym import spaces

from IPython.display import clear_output
import time
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
@dataclass
class SCurveConfig:
    dt: float = 0.05
    episode_seconds: float = 12.0

    # Dynamics limits
    a_max: float = 3.0          # max acceleration magnitude per axis
    v_max: float = 4.0          # max speed magnitude

    # Path geometry (S-curve)
    x_start: float = 0.0
    x_goal: float = 10.0
    y_center: float = 0.0
    y_amp: float = 2.0
    s_turns: float = 1.0        # number of sine periods over the path length

    # Reward: negative absolute tracking error (no weights needed)

    # Termination thresholds
    goal_radius: float = 0.25
    max_track_error: float = 4.0  # terminate if too far from path

    # Observation: how many reference points ahead to include
    lookahead_k: int = 2         # include k future ref points
    lookahead_dt: float = 0.25   # time gap for each future ref point

    # Reset noise
    pos_noise: float = 0.05
    vel_noise: float = 0.05


class SCurve2DEnv(gym.Env):
    """
    2D continuous control env:
      state: [x, y, vx, vy]
      action: [ax, ay] clipped to [-a_max, a_max]
      dynamics: v <- clip(v + a*dt, v_max), p <- p + v*dt

    Reference S-curve is parameterized by "progress" u in [0,1] mapped to x in [x_start, x_goal].
    Reward is the negative absolute distance from the time-based reference point on the curve.
    """

    metadata = {"render_modes": ["human"], "render_fps": 30}

    def __init__(self, config: SCurveConfig = SCurveConfig(), render_mode=None):
        super().__init__()
        self.cfg = config
        self.render_mode = render_mode

        self.dt = self.cfg.dt
        self.T = self.cfg.episode_seconds
        self.max_steps = int(np.ceil(self.T / self.dt))

        # Action space: 2D acceleration
        self.action_space = spaces.Box(
            low=-self.cfg.a_max, high=self.cfg.a_max, shape=(2,), dtype=np.float32
        )

        # Observation:
        # [x, y, vx, vy, x_goal, y_goal, x_ref0, y_ref0, x_ref1, y_ref1, ...]
        obs_dim = 4 + 2 + 2 * (1 + self.cfg.lookahead_k)
        high = np.full((obs_dim,), np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(-high, high, dtype=np.float32)

        # Internal state
        self.state = None
        self.step_count = 0
        self.t = 0.0
        self.u = 0.0  # progress estimate [0,1] based on x position

        # Rendering
        self._fig = None
        self._ax = None
        self._artist_agent = None
        self._artist_path = None
        self._artist_ref = None

    # ---------- Path definition ----------
    def _path_xy(self, u: float) -> np.ndarray:
        """
        u in [0,1] -> (x,y) along S-curve
        x moves linearly start->goal
        y = y_center + y_amp * sin(2*pi*s_turns*u)
        """
        u = np.clip(u, 0.0, 1.0)
        x = self.cfg.x_start + (self.cfg.x_goal - self.cfg.x_start) * u
        y = self.cfg.y_center + self.cfg.y_amp * np.sin(2.0 * np.pi * self.cfg.s_turns * u)
        return np.array([x, y], dtype=np.float32)

    def _path_tangent(self, u: float) -> np.ndarray:
        """
        Tangent vector d(x,y)/du (not normalized)
        """
        u = np.clip(u, 0.0, 1.0)
        dx_du = (self.cfg.x_goal - self.cfg.x_start)
        dy_du = self.cfg.y_amp * (2.0 * np.pi * self.cfg.s_turns) * np.cos(2.0 * np.pi * self.cfg.s_turns * u)
        return np.array([dx_du, dy_du], dtype=np.float32)

    def _path_curvature(self, u: float) -> float:
        """Signed curvature κ of the S-curve at progress u."""
        u = np.clip(u, 0.0, 1.0)
        L = self.cfg.x_goal - self.cfg.x_start
        omega = 2.0 * np.pi * self.cfg.s_turns
        dy_du = self.cfg.y_amp * omega * np.cos(omega * u)
        d2y_du2 = -self.cfg.y_amp * omega ** 2 * np.sin(omega * u)
        return abs(L * d2y_du2) / ((L ** 2 + dy_du ** 2) ** 1.5 + 1e-8)

    def _estimate_progress_from_pos(self, pos: np.ndarray) -> float:
        """
        Simple progress proxy: map x position to u.
        This is not exact projection to curve, but works well for this scenario and is fast.
        """
        x = pos[0]
        u = (x - self.cfg.x_start) / (self.cfg.x_goal - self.cfg.x_start)
        return float(np.clip(u, 0.0, 1.0))

    # ---------- Gym API ----------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.t = 0.0

        start = self._path_xy(0.0)
        # Add small noise so BC/DAgger differences show up
        pos = start + self.np_random.normal(0.0, self.cfg.pos_noise, size=(2,)).astype(np.float32)
        vel = self.np_random.normal(0.0, self.cfg.vel_noise, size=(2,)).astype(np.float32)

        self.state = np.concatenate([pos, vel]).astype(np.float32)
        self.u = self._estimate_progress_from_pos(pos)

        obs = self._get_obs()
        info = self._get_info()
        if self.render_mode == "human":
            self.render()
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -self.cfg.a_max, self.cfg.a_max)

        x, y, vx, vy = self.state

        # Update velocity
        vx = vx + action[0] * self.dt
        vy = vy + action[1] * self.dt

        # Clip speed
        v = np.array([vx, vy], dtype=np.float32)
        speed = np.linalg.norm(v) + 1e-8
        if speed > self.cfg.v_max:
            v = v * (self.cfg.v_max / speed)
        vx, vy = float(v[0]), float(v[1])

        # Update position
        x = x + vx * self.dt
        y = y + vy * self.dt

        self.state = np.array([x, y, vx, vy], dtype=np.float32)

        # Time/progress
        self.step_count += 1
        self.t = self.step_count * self.dt
        self.u = self._estimate_progress_from_pos(self.state[:2])

        # Compute reward
        ref = self._path_xy(self._u_ref())
        track_err = np.linalg.norm(self.state[:2] - ref)

        reward = -track_err

        # Termination conditions
        goal = self._path_xy(1.0)
        dist_to_goal = np.linalg.norm(self.state[:2] - goal)
        terminated = dist_to_goal <= self.cfg.goal_radius
        truncated = (self.step_count >= self.max_steps)

        # Safety terminate if agent flies away
        if track_err > self.cfg.max_track_error:
            truncated = True  # treat as truncation (or terminated if you prefer)

        obs = self._get_obs()
        info = self._get_info()
        info.update({
            "track_err": float(track_err),
            "u": float(self.u),
            "u_ref": float(self._u_ref()),
            "dist_to_goal": float(dist_to_goal),
        })

        if self.render_mode == "human":
            self.render()

        return obs, reward, terminated, truncated, info

    def _u_ref(self) -> float:
        """Time-based reference progress."""
        return float(np.clip(self.t / self.T, 0.0, 1.0))

    def _get_obs(self) -> np.ndarray:
        pos = self.state[:2]
        vel = self.state[2:]
        goal = self._path_xy(1.0)

        # include current ref and future refs (lookahead)
        refs = [self._path_xy(self._u_ref())]
        for k in range(1, self.cfg.lookahead_k + 1):
            u_k = np.clip((self.t + k * self.cfg.lookahead_dt) / self.T, 0.0, 1.0)
            refs.append(self._path_xy(u_k))
        refs = np.concatenate(refs, axis=0)

        obs = np.concatenate([pos, vel, goal, refs]).astype(np.float32)
        return obs

    def _get_info(self):
        return {}

    # ---------- Simple expert policy ----------
    def expert_policy(self, obs: np.ndarray) -> np.ndarray:
        """
        Curvature-aware PD tracking controller with lookahead blending.

        Improvements over a basic PD tracker:
        - Blended position target from all reference points (near for
          cross-track correction, far for anticipation).
        - Curvature-adaptive desired speed: slows on tight curves to
          reduce overshoot, speeds up on straight segments.
        - Goal deceleration to avoid overshooting the goal.
        - Higher derivative gain for reduced oscillation.
        """
        obs = np.asarray(obs, dtype=np.float32)
        pos = obs[0:2]
        vel = obs[2:4]
        goal = obs[4:6]

        refs_flat = obs[6:]
        refs = refs_flat.reshape(-1, 2)
        far_ref = refs[-1]

        # Blended target: linearly increasing weights toward the far ref
        n = len(refs)
        w = np.arange(1.0, n + 1.0)
        w /= w.sum()
        target = (refs * w[:, None]).sum(axis=0)

        u_far = self._estimate_progress_from_pos(far_ref)

        # Curvature-adaptive desired speed
        curvature = self._path_curvature(u_far)
        v_base = 2.5
        v_des_mag = v_base / (1.0 + 10.0 * curvature)

        # Decelerate when approaching the goal
        dist_to_goal = np.linalg.norm(goal - pos)
        if dist_to_goal < 1.5:
            v_des_mag *= float(np.clip(dist_to_goal / 1.5, 0.2, 1.0))

        # Desired velocity along path tangent at far lookahead
        tan = self._path_tangent(u_far)
        direction = tan / (np.linalg.norm(tan) + 1e-8)
        v_des = direction * v_des_mag

        kp = 6.0
        kd = 3.0
        a = kp * (target - pos) + kd * (v_des - vel)

        a = np.clip(a, -self.cfg.a_max, self.cfg.a_max).astype(np.float32)
        return a

    # ---------- Rendering ----------
    def render(self):
        import matplotlib.pyplot as plt

        if self._fig is None:
            self._fig, self._ax = plt.subplots(figsize=(6, 4))
            self._ax.set_aspect("equal", adjustable="box")
            self._ax.set_title("SCurve2DEnv")
            self._ax.set_xlabel("x")
            self._ax.set_ylabel("y")

            # draw reference path
            us = np.linspace(0, 1, 200)
            path = np.stack([self._path_xy(u) for u in us], axis=0)
            (self._artist_path,) = self._ax.plot(path[:, 0], path[:, 1], linewidth=2)

            # agent point
            (self._artist_agent,) = self._ax.plot([], [], marker="o", markersize=8)

            # current ref point
            (self._artist_ref,) = self._ax.plot([], [], marker="x", markersize=8)

            self._ax.set_xlim(self.cfg.x_start - 1.0, self.cfg.x_goal + 1.0)
            self._ax.set_ylim(self.cfg.y_center - (self.cfg.y_amp + 3.0), self.cfg.y_center + (self.cfg.y_amp + 3.0))

        pos = self.state[:2]
        ref = self._path_xy(self._u_ref())

        self._artist_agent.set_data([pos[0]], [pos[1]])
        self._artist_ref.set_data([ref[0]], [ref[1]])

        self._ax.figure.canvas.draw()
        self._ax.figure.canvas.flush_events()

    def close(self):
        if self._fig is not None:
            import matplotlib.pyplot as plt
            plt.close(self._fig)
            self._fig = None
            self._ax = None



In [ ]:
env = SCurve2DEnv(render_mode=None)
obs, info = env.reset(seed=0)
total = 0.0
for _ in range(200):
    a = env.expert_policy(obs)
    obs, r, term, trunc, info = env.step(a)
    total += r
    if term or trunc:
        break
print("done:", term, trunc, "steps:", env.step_count, "u:", info.get("u"), "total reward:", total)

In [ ]:
from matplotlib import lines


env = SCurve2DEnv(render_mode=None)
obs, _ = env.reset(seed=0)

states = []

for t in tqdm(range(400), desc="Collecting expert trajectory"):
    action = env.expert_policy(obs)
    obs, r, terminated, truncated, info = env.step(action)
    states.append(env.state[:2].copy())
    if terminated or truncated:
        print("Episode ended at step", t, "terminated:", terminated, "truncated:", truncated)
        break

states = np.array(states)

# Reference path
us = np.linspace(0, 1, 300)
ref_path = np.stack([env._path_xy(u) for u in us])

plt.figure(figsize=(6,4))
plt.plot(ref_path[:,0], ref_path[:,1], linewidth=2, label="Reference S-curve",linestyle="--")
plt.plot(states[:,0], states[:,1], linewidth=2, label="Agent trajectory", linestyle="--")
plt.scatter(states[0,0], states[0,1], s=80, label="Start")
plt.scatter(states[-1,0], states[-1,1], s=80, label="End")
plt.legend()
plt.axis("equal")
plt.title("Expert Trajectory Following S-curve")
plt.show()

## Behavior Cloning

**Behavior Cloning (BC)** is the simplest imitation learning algorithm.  
We treat the problem as pure supervised learning:

1. **Collect** a dataset of (observation, action) pairs from the expert policy.  
2. **Train** a neural-network policy $\pi_\theta(o)$ to minimize $\| \pi_\theta(o) - a_{\text{expert}} \|^2$.  
3. **Evaluate** the learned policy in the environment and compare it to the expert.

A key limitation of BC is **distribution shift**: during training the learner only sees states the *expert* visits, but at test time the learner's own small errors compound and push it into states it has never seen.

### Step 1 — Collect Expert Demonstrations

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def collect_expert_data(env, n_episodes=50, seed_start=0):
    """Roll out the expert policy and store (obs, action) pairs."""
    all_obs, all_act = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed_start + ep)
        done = False
        while not done:
            # TODO: use the expert policy to get action, store obs and action, step the env, and check termination, return the collected data as numpy arrays
            raise NotImplementedError()

    return None, None

env = SCurve2DEnv(render_mode=None)
expert_obs, expert_act = collect_expert_data(env, n_episodes=50)
print(f"Collected {len(expert_obs)} transitions from {50} episodes")
print(f"Observation shape: {expert_obs.shape},  Action shape: {expert_act.shape}")

### Step 2 — Define the BC Policy Network

In [ ]:
class BCPolicy(nn.Module):
    """Simple MLP policy: obs -> action."""

    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: define a simple MLP with 2 hidden layers and ReLU activations that maps obs_dim to act_dim

        )

    def forward(self, obs):
        return self.net(obs)

obs_dim = expert_obs.shape[1]
act_dim = expert_act.shape[1]

bc_policy = BCPolicy(obs_dim, act_dim)
print(bc_policy)
print(f"Parameters: {sum(p.numel() for p in bc_policy.parameters()):,}")

### Step 3 — Train the BC Policy

In [ ]:
def train_bc(policy, obs, act, epochs=80, batch_size=256, lr=1e-3):
    """Train policy with MSE loss on expert (obs, act) pairs."""
    dataset = TensorDataset(torch.tensor(obs), torch.tensor(act))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = optim.Adam(policy.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    history = []
    for epoch in tqdm(range(1, epochs + 1), desc="Training BC policy"):
        epoch_loss = 0.0
        n_batches = 0
        for ob, ac in loader:
            pred = policy(ob)
            loss = loss_fn(pred, ac)
            #TODO - perform optimization step to update policy parameters based on the loss

            epoch_loss += loss.item()
            n_batches += 1
        avg = #TODO - compute average loss for the epoch
        history.append(avg)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}/{epochs}  loss = {avg:.6f}")
    return history

bc_loss_history = train_bc(bc_policy, expert_obs, expert_act)

In [ ]:
# Plot BC training loss curve

plt.figure(figsize=(6, 3))
plt.plot(bc_loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("BC Training Loss")
plt.tight_layout()
plt.show()

### Step 4 — Evaluate the BC Policy

In [ ]:
@torch.no_grad()
def rollout(env, policy_fn, seed=0, max_steps=500):
    """Roll out a policy and return states, rewards, info."""
    obs, _ = env.reset(seed=seed)
    states, rewards = [env.state[:2].copy()], []
    for _ in range(max_steps):
        action = policy_fn(obs)
        obs, r, terminated, truncated, info = env.step(action)
        states.append(env.state[:2].copy())
        rewards.append(r)
        if terminated or truncated:
            break
    return np.array(states), np.sum(rewards), info

def bc_policy_fn(obs):
    """Wraps the PyTorch BC policy for use with env.step."""
    t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
    a = bc_policy(t).squeeze(0).numpy()
    return np.clip(a, -env.cfg.a_max, env.cfg.a_max)

env = SCurve2DEnv(render_mode=None)

n_eval = 20
expert_rewards, bc_rewards = [], []
for s in range(n_eval):
    _, er, _ = rollout(env, env.expert_policy, seed=1000 + s)
    _, br, _ = rollout(env, bc_policy_fn, seed=1000 + s)
    expert_rewards.append(er)
    bc_rewards.append(br)

print(f"Expert  — mean reward: {np.mean(expert_rewards):.2f} ± {np.std(expert_rewards):.2f}")
print(f"BC      — mean reward: {np.mean(bc_rewards):.2f} ± {np.std(bc_rewards):.2f}")

### Step 5 — Visualize: Expert vs BC Trajectories

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Reference path for all subplots
us = np.linspace(0, 1, 300)
ref_path = np.stack([env._path_xy(u) for u in us])

seeds_to_show = [1000, 1001, 1002]
for ax, seed in zip(axes, seeds_to_show):
    expert_states, er, _ = rollout(env, env.expert_policy, seed=seed)
    bc_states, br, _ = rollout(env, bc_policy_fn, seed=seed)

    ax.plot(ref_path[:, 0], ref_path[:, 1], "k--", linewidth=1, alpha=0.4, label="Reference")
    ax.plot(expert_states[:, 0], expert_states[:, 1], linewidth=2, label=f"Expert ({er:.1f})")
    ax.plot(bc_states[:, 0], bc_states[:, 1], linewidth=2, label=f"BC ({br:.1f})")
    ax.set_title(f"Seed {seed}")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(fontsize=8)

fig.suptitle("Behavioral Cloning vs Expert", fontsize=14)
plt.tight_layout()
plt.show()

## Theory Check


### Question:
The expert demonstrations were collected with tiny initial noise (`pos_noise = 0.05`, `vel_noise = 0.05`), so the BC policy has only ever seen states that are very close to the reference curve. What happens when the agent starts from a **radically perturbed** initial position?

### Answer:

### Empirical Check

 The experiment below deliberately start far from the training curve. Do they confirm your answer?

In [ ]:
@torch.no_grad()
def rollout_perturbed(env, policy_fn, seed=0, pos_offset=None, vel_offset=None, max_steps=500):
    """Like rollout(), but injects a large perturbation into the initial state."""
    obs, _ = env.reset(seed=seed)
    if pos_offset is not None:
        env.state[0] += pos_offset[0]
        env.state[1] += pos_offset[1]
    if vel_offset is not None:
        env.state[2] += vel_offset[0]
        env.state[3] += vel_offset[1]
    env.u = env._estimate_progress_from_pos(env.state[:2])
    obs = env._get_obs()

    states, rewards = [env.state[:2].copy()], []
    for _ in range(max_steps):
        action = policy_fn(obs)
        obs, r, terminated, truncated, info = env.step(action)
        states.append(env.state[:2].copy())
        rewards.append(r)
        if terminated or truncated:
            break
    return np.array(states), np.sum(rewards), info

env = SCurve2DEnv(render_mode=None)
us_ref = np.linspace(0, 1, 300)
ref_path = np.stack([env._path_xy(u) for u in us_ref])

perturbations = [
    {"label": "Moderate (y +1.5)",   "pos": np.array([0.0,  1.5]), "vel": None},
    {"label": "Large (y -3.0)",      "pos": np.array([0.0, -1.5]), "vel": None},
    {"label": "Extreme (y +4, vx +2)", "pos": np.array([0.0, 2.0]), "vel": np.array([2.0, 0.0])},
]

fig, axes = plt.subplots(1, len(perturbations), figsize=(6 * len(perturbations), 5))

for ax, pert in zip(axes, perturbations):
    expert_states, er, _ = rollout_perturbed(
        env, env.expert_policy, seed=42,
        pos_offset=pert["pos"], vel_offset=pert["vel"],
    )
    bc_states, br, _ = rollout_perturbed(
        env, bc_policy_fn, seed=42,
        pos_offset=pert["pos"], vel_offset=pert["vel"],
    )

    ax.plot(ref_path[:, 0], ref_path[:, 1], "k--", lw=1, alpha=0.4, label="Reference")
    ax.plot(expert_states[:, 0], expert_states[:, 1], lw=2, label=f"Expert ({er:.1f})")
    ax.plot(bc_states[:, 0], bc_states[:, 1], lw=2, label=f"BC ({br:.1f})")
    ax.scatter(*expert_states[0], marker="o", s=100, zorder=5, label="Start")
    ax.set_title(pert["label"], fontsize=11)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(fontsize=8, loc="best")

fig.suptitle("BC vs Expert — Perturbed Initial Conditions", fontsize=14)
plt.tight_layout()
plt.show()

### Quantitative Comparison Across Many Perturbed Seeds

Below we sweep over 20 random seeds with a **fixed large perturbation** and report mean/std reward for both policies. This confirms the failure is systematic, not a one-off fluke.

In [ ]:
large_offset = np.array([0.0, -3.0])
n_eval = 20

expert_rew_perturbed, bc_rew_perturbed = [], []
for s in range(n_eval):
    _, er, _ = rollout_perturbed(env, env.expert_policy, seed=3000 + s, pos_offset=large_offset)
    _, br, _ = rollout_perturbed(env, bc_policy_fn, seed=3000 + s, pos_offset=large_offset)
    expert_rew_perturbed.append(er)
    bc_rew_perturbed.append(br)

print("=== Perturbed initial conditions (y offset = -3.0) ===")
print(f"Expert — mean reward: {np.mean(expert_rew_perturbed):.2f} ± {np.std(expert_rew_perturbed):.2f}")
print(f"BC     — mean reward: {np.mean(bc_rew_perturbed):.2f} ± {np.std(bc_rew_perturbed):.2f}")
print()
print("=== Normal initial conditions (for reference) ===")
print(f"Expert — mean reward: {np.mean(expert_rewards):.2f} ± {np.std(expert_rewards):.2f}")
print(f"BC     — mean reward: {np.mean(bc_rewards):.2f} ± {np.std(bc_rewards):.2f}")
print()
gap_normal = np.mean(expert_rewards) - np.mean(bc_rewards)
gap_perturbed = np.mean(expert_rew_perturbed) - np.mean(bc_rew_perturbed)
print(f"Reward gap (expert - BC):  normal = {gap_normal:.2f},  perturbed = {gap_perturbed:.2f}")
print(f"The gap widens by {gap_perturbed / max(gap_normal, 1e-8):.1f}x under perturbation.")

## Theory Check - Noisy Execution Problem

### Questions:

In practice, it can also happen that even starting from a non perturbated position, the controller may be subject to small disturbances at every step (e.g., actuator noise).
- What happens when can happen when such small but persistent perturbations are introduced during execution?
- What's a potential fix to this issue? 

### Answers:

### Empirical Check

The experiment below simulates that the agent's velocity is randomly perturbed by Gaussian noise ($\sigma = 0.1$). Do the reuslts confirm your intuition?

In [ ]:
VEL_NOISE_STD = 0.1

@torch.no_grad()
def rollout_noisy(env, policy_fn, seed=0, vel_noise_std=VEL_NOISE_STD, max_steps=500):
    """Roll out a policy with velocity noise injected at every transition."""
    obs, _ = env.reset(seed=seed)
    rng = np.random.RandomState(seed + 10_000)
    states, rewards = [env.state[:2].copy()], []
    for _ in range(max_steps):
        action = policy_fn(obs)
        obs, r, terminated, truncated, info = env.step(action)
        env.state[2:4] += rng.normal(0, vel_noise_std, size=2).astype(np.float32)
        obs = env._get_obs()
        states.append(env.state[:2].copy())
        rewards.append(r)
        if terminated or truncated:
            break
    return np.array(states), np.sum(rewards), info

# --- Evaluate BC under noisy execution ---
env = SCurve2DEnv(render_mode=None)
n_eval = 20

expert_noisy_rews, bc_noisy_rews = [], []
for s in range(n_eval):
    _, er, _ = rollout_noisy(env, env.expert_policy, seed=2000 + s)
    _, br, _ = rollout_noisy(env, bc_policy_fn, seed=2000 + s)
    expert_noisy_rews.append(er)
    bc_noisy_rews.append(br)

print(f"=== Noisy execution (vel_noise_std = {VEL_NOISE_STD}) ===")
print(f"Expert — mean reward: {np.mean(expert_noisy_rews):.2f} ± {np.std(expert_noisy_rews):.2f}")
print(f"BC     — mean reward: {np.mean(bc_noisy_rews):.2f} ± {np.std(bc_noisy_rews):.2f}")
print()
print("=== Clean execution (for reference) ===")
print(f"Expert — mean reward: {np.mean(expert_rewards):.2f} ± {np.std(expert_rewards):.2f}")
print(f"BC     — mean reward: {np.mean(bc_rewards):.2f} ± {np.std(bc_rewards):.2f}")
print()
gap_clean = abs(np.mean(expert_rewards) - np.mean(bc_rewards))
gap_noisy = abs(np.mean(expert_noisy_rews) - np.mean(bc_noisy_rews))
print(f"Reward gap |expert − BC|:  clean = {gap_clean:.2f},  noisy = {gap_noisy:.2f}")
if gap_clean > 0:
    print(f"The gap widens by {gap_noisy / gap_clean:.1f}x under noisy execution.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
us = np.linspace(0, 1, 300)
ref_path = np.stack([env._path_xy(u) for u in us])

for ax, seed in zip(axes, [2000, 2001, 2002]):
    expert_states, er, _ = rollout_noisy(env, env.expert_policy, seed=seed)
    bc_states, br, _ = rollout_noisy(env, bc_policy_fn, seed=seed)

    ax.plot(ref_path[:, 0], ref_path[:, 1], "k--", lw=1, label="Reference")
    ax.plot(expert_states[:, 0], expert_states[:, 1], lw=2, label=f"Expert ({er:.1f})")
    ax.plot(bc_states[:, 0], bc_states[:, 1], lw=2, label=f"BC ({br:.1f})")
    ax.legend(fontsize=8)
    ax.set_title(f"Seed {seed}")
    ax.axis("equal")

fig.suptitle(
    f"BC vs Expert — Noisy Execution (vel noise σ = {VEL_NOISE_STD})", fontsize=14
)
plt.tight_layout()
plt.show()

## DAgger: Closing the Distribution Gap

**DAgger (Dataset Aggregation)** [Ross et al., 2011] iteratively solves the distribution shift problem:

| Step | Description |
|------|-------------|
| **1. Train** | Train a policy $\hat\pi_i$ on the current aggregated dataset $\mathcal{D}_i$. |
| **2. Roll out** | Execute $\hat\pi_i$ in the environment (with the same execution noise) to collect states $s_1, s_2, \ldots$ the *learner* actually visits. |
| **3. Label** | Query the expert $\pi^*$ for the correct action $a^*_t = \pi^*(s_t)$ at each visited state. |
| **4. Aggregate** | $\mathcal{D}_{i+1} = \mathcal{D}_i \cup \{(s_t, a^*_t)\}$. |

Repeat for several iterations. The key insight is that the training distribution progressively covers the states the learner encounters at *test time*, so the compounding error shrinks with each iteration.

In [ ]:
def _train_silent(policy, obs_np, act_np, epochs=40, batch_size=256, lr=1e-3):
    """Same as train_bc but without printing and returning only the final loss (used inside DAgger loop)."""
    dataset = TensorDataset(torch.tensor(obs_np), torch.tensor(act_np))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    final_loss = 0.0
    for epoch in range(1, epochs + 1):
        epoch_loss, n = 0.0, 0
        for ob, ac in loader:
            loss = loss_fn(policy(ob), ac)
            #TODO - perform optimization step to update policy parameters based on the loss
            epoch_loss += loss.item()
            n += 1
        final_loss = epoch_loss / n
    return final_loss


N_DAGGER_ITERS = 6
N_EPISODES_PER_ITER = 20
DAGGER_EPOCHS = 40

env = SCurve2DEnv(render_mode=None)

# Seed the aggregated dataset with the original expert demonstrations
agg_obs = [expert_obs.copy()]
agg_act = [expert_act.copy()]

dagger_policy = None #TODO - Instantiate the BCPolicy model
dagger_iter_rewards = []

print(f"Running DAgger ({N_DAGGER_ITERS} iterations, "
      f"{N_EPISODES_PER_ITER} episodes/iter, vel_noise σ = {VEL_NOISE_STD})\n")

for it in range(N_DAGGER_ITERS):
    # 1. Train on the full aggregated dataset
    all_obs_np = np.concatenate(agg_obs, axis=0)
    all_act_np = np.concatenate(agg_act, axis=0)
    final_loss = _train_silent(dagger_policy, all_obs_np, all_act_np,
                               epochs=DAGGER_EPOCHS)

    # Wrapped policy for rollout
    @torch.no_grad()
    def dagger_policy_fn(obs):
        t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        return np.clip(dagger_policy(t).squeeze(0).numpy(),
                       -env.cfg.a_max, env.cfg.a_max)

    # 2. Roll out the learner WITH noise; record every visited state
    new_obs, new_act = [], []
    iter_rews = []
    for ep in range(N_EPISODES_PER_ITER):
        seed = 5000 + it * N_EPISODES_PER_ITER + ep
        obs, _ = env.reset(seed=seed)
        rng = np.random.RandomState(seed + 20_000)
        ep_rew, done = 0.0, False
        while not done:
            # 3. Label the current state with the expert's action
            expert_action = #TODO - get expert action for the current observation
            new_obs.append(None) #TODO - Replace
            new_act.append(None) #TODO - Replace

            # Execute the learner's own action
            action = #TODO - get action from the learner policy 
            obs, r, term, trunc, _ = env.step(action)

            # Inject velocity noise 
            env.state[2:4] += rng.normal(0, VEL_NOISE_STD,
                                         size=2).astype(np.float32)
            obs = env._get_obs()

            ep_rew += r
            done = term or trunc
        iter_rews.append(ep_rew)

    # 4. Aggregate
    agg_obs.append(None) #TODO - Replace 
    agg_act.append(None) #TODO - Replace

    mean_rew = np.mean(iter_rews)
    dagger_iter_rewards.append(mean_rew)
    total_samples = sum(len(o) for o in agg_obs)
    print(f"  iter {it+1}/{N_DAGGER_ITERS}  "
          f"new: {len(new_obs):>5d}  total: {total_samples:>6d}  "
          f"loss: {final_loss:.4f}  mean reward: {mean_rew:.2f}")

print("\nDAgger training complete!")

In [ ]:
# --- Final quantitative comparison under noisy execution ---
n_eval = 20
dagger_noisy_rews = []
for s in range(n_eval):
    _, dr, _ = rollout_noisy(env, dagger_policy_fn, seed=2000 + s)
    dagger_noisy_rews.append(dr)

print(f"=== Final comparison — noisy execution (σ = {VEL_NOISE_STD}) ===")
print(f"Expert — mean reward: {np.mean(expert_noisy_rews):>8.2f} ± {np.std(expert_noisy_rews):.2f}")
print(f"BC     — mean reward: {np.mean(bc_noisy_rews):>8.2f} ± {np.std(bc_noisy_rews):.2f}")
print(f"DAgger — mean reward: {np.mean(dagger_noisy_rews):>8.2f} ± {np.std(dagger_noisy_rews):.2f}")
print()
bc_gap = np.mean(expert_noisy_rews) - np.mean(bc_noisy_rews)
dag_gap = np.mean(expert_noisy_rews) - np.mean(dagger_noisy_rews)
print(f"Gap to expert:  BC = {bc_gap:.2f},  DAgger = {dag_gap:.2f}")
if abs(bc_gap) > 1e-6:
    print(f"DAgger closes {100 * (1 - dag_gap / bc_gap):.0f}% of the BC→Expert gap.")

# --- Trajectory visualization: Expert vs BC vs DAgger ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
us = np.linspace(0, 1, 300)
ref_path = np.stack([env._path_xy(u) for u in us])

for ax, seed in zip(axes, [2000, 2001, 2002]):
    exp_s, er, _ = rollout_noisy(env, env.expert_policy, seed=seed)
    bc_s, br, _  = rollout_noisy(env, bc_policy_fn, seed=seed)
    dag_s, dr, _ = rollout_noisy(env, dagger_policy_fn, seed=seed)

    ax.plot(ref_path[:, 0], ref_path[:, 1], "k--", lw=1, label="Reference")
    ax.plot(exp_s[:, 0], exp_s[:, 1], lw=2, label=f"Expert ({er:.1f})")
    ax.plot(bc_s[:, 0], bc_s[:, 1], lw=2, label=f"BC ({br:.1f})")
    ax.plot(dag_s[:, 0], dag_s[:, 1], lw=2, label=f"DAgger ({dr:.1f})")
    ax.legend(fontsize=8)
    ax.set_title(f"Seed {seed}")
    ax.axis("equal")

fig.suptitle(
    f"Expert vs BC vs DAgger — Noisy Execution (σ = {VEL_NOISE_STD})",
    fontsize=14,
)
plt.tight_layout()
plt.show()

# --- DAgger learning curve ---
fig, ax = plt.subplots(figsize=(7, 3.5))
iters = range(1, N_DAGGER_ITERS + 1)
ax.plot(iters, dagger_iter_rewards, "o-", lw=2, label="DAgger (noisy rollout)")
ax.axhline(np.mean(bc_noisy_rews), color="red", ls="--", lw=1.5,
           label=f"BC baseline ({np.mean(bc_noisy_rews):.1f})")
ax.axhline(np.mean(expert_noisy_rews), color="green", ls="--", lw=1.5,
           label=f"Expert ({np.mean(expert_noisy_rews):.1f})")
ax.set_xlabel("DAgger Iteration")
ax.set_ylabel("Mean Reward (noisy execution)")
ax.set_title("DAgger Training Progress")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()